In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
class ProbingHead(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super(ProbingHead, self).__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2) 
        )
    def forward(self, h):
        return self.mlp(h)

class HallucinationDetector(nn.Module):
    def __init__(self, num_layers, hidden_size, hidden_dim=256):
        super(HallucinationDetector, self).__init__()
        self.num_layers = num_layers
        self.probing_heads = nn.ModuleList([
            ProbingHead(hidden_size, hidden_dim) for _ in range(num_layers)
        ])

    def forward(self, hidden_states):
        logits = []
        for l in range(self.num_layers):
            logits_l = self.probing_heads[l](hidden_states[l])
            logits.append(logits_l)
        logits = torch.stack(logits, dim=0)
        logits = logits.mean(dim=0)
        z_i = F.softmax(logits, dim=-1)
        return z_i

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.gamma = gamma

    def forward(self, probs, targets):
        pt = probs[:, :, 1]
        targets = targets.float()
        loss = - (1-pt)**self.gamma*torch.log(pt+1e-8)*targets
        return loss.mean()

In [ ]:
import torch
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, AdamW

class HallucinationDataset(Dataset):
    def __init__(self, prompts, labels, tokenizer):
        self.inputs = [tokenizer(p, return_tensors="pt", padding='max_length', truncation=True, max_length=128) for p in prompts]
        self.labels = labels

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        return self.inputs[idx], self.labels[idx]

def train_detector(detector, model, tokenizer, dataloader, device):
    detector.train()
    model.eval()
    loss_fn = FocalLoss(gamma=2.0)
    optimizer = AdamW(detector.parameters(), lr=1e-4)

    for epoch in range(10):
        total_loss = 0.0
        for batch in dataloader:
            input_batch, label_batch = batch
            input_batch = {k: v.squeeze(0).to(device) for k, v in input_batch.items()}
            label_batch = label_batch.to(device)

            with torch.no_grad():
                outputs = model(**input_batch, output_hidden_states=True)
                hidden_states = outputs.hidden_states[1:]  # [L, B, T, H]

            hidden_states = [h.detach() for h in hidden_states]
            hidden_states = [h.to(device) for h in hidden_states]

            probs = detector(hidden_states)  # [B, T, 2]
            loss = loss_fn(probs, label_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}: Loss = {total_loss / len(dataloader):.4f}")



In [7]:
# model_name = "Qwen/Qwen3-1.7B"
# model = AutoModelForCausalLM.from_pretrained(model_name, output_hidden_states=True).eval()
# tokenizer = AutoTokenizer.from_pretrained(model_name)